# AnyMatch — Sequential fine-tune on the synthetic AllianceChicago corpus (Colab A100)

Resumes from the zero-shot **mode4** checkpoint (`anymatch_all9_gpt2_mode4`, trained on the 9 public EM datasets by `anymatch_training.ipynb`) and **continues training** on the synthetic FQHC pair corpus. The goal is to teach the two behaviours the zero-shot model misses — **SSN as decisive identity** and **cross-name-field token shuffles** — without forgetting general EM ability.

This is the *sequential* (not joint-mix) fine-tune: gentle LR + early stopping on the entity-disjoint held-out test.

**Before running**: Runtime → Change runtime type → **A100 GPU**.

**Consistency guarantee**: training serializes via `utils/alliance_schema.py`, the same module both inference notebooks import — so the model is trained and served on identical attribute names (three separate name fields, suffix, address2, ssn + ssn4, email).

**PHI reminder**: the synthetic corpus is non-PHI, but keep this on Colab Pro / Workspace per your institution's policy.

## 1. Mount Drive and bring in the code

Same workflow as `anymatch_training.ipynb`: zip the local `AnyMatch/` folder (patches + `utils/alliance_schema.py` + `finetune_alliance.py` + `data/synthetic/finetune_*.csv` included) and drop `AnyMatch.zip` in your Drive root.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, shutil
WORK = '/content/anymatch'
DRIVE_ZIP = '/content/drive/MyDrive/AnyMatch.zip'
DRIVE_CKPT_DIR = '/content/drive/MyDrive/AnyMatch/checkpoints'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

!rm -rf {WORK} && mkdir -p {WORK}
!unzip -q {DRIVE_ZIP} -d {WORK}

# Flatten one level if the zip nested everything under AnyMatch/.
inner = glob.glob(f'{WORK}/*/loo.py')
if inner:
    src = os.path.dirname(inner[0])
    !cp -a {src}/. {WORK}/
    shutil.rmtree(src, ignore_errors=True)

%cd {WORK}
!ls

Mounted at /content/drive
/content/anymatch
anymatch_alliance_inference.ipynb   inference.py
anymatch_finetuning.ipynb	    loo.py
anymatch.png			    __MACOSX
anymatch_synthetic_inference.ipynb  model.py
anymatch_training.ipynb		    predict_alliance.py
CLAUDE.md			    __pycache__
data				    README.md
data.py				    saved_models
diagram				    SETUP.md
docs				    string_simlarity.py
environment.yml			    synthetic_data_generation
finetune_alliance.py		    utils


## 2. Install dependencies

Same pinned stack as training. `finetune_alliance.py` uses `one_pos_two_neg`-free data (the synthetic CSVs are pre-balanced), so **no autogluon needed**.

In [2]:
# !pip install -q 'transformers==4.36.2' 'tokenizers<0.20' 'datasets<3' 'scikit-learn' 'pandas' 'tqdm'
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

CUDA available: True | NVIDIA A100-SXM4-40GB


## 3. Verify the synthetic corpus is present

The fine-tune needs the generated pair CSVs in `data/synthetic/`:
- `finetune_train_v1.csv` — case-first balanced training set
- `finetune_test_v1.csv` — entity-disjoint held-out (drives early stopping)
- `realistic_eval_v1.csv` — realistic 1:9 distribution (final report-only metric)

Fallback: sync from `MyDrive/AnyMatch/synthetic/` if they weren't in the zip.

In [3]:
LOCAL_SYNTH = 'data/synthetic'
DRIVE_SYNTH = '/content/drive/MyDrive/AnyMatch/synthetic'
VERSION = 'v1'
need = [f'finetune_train_{VERSION}.csv', f'finetune_test_{VERSION}.csv', f'realistic_eval_{VERSION}.csv']

have = all(os.path.exists(os.path.join(LOCAL_SYNTH, f)) for f in need)
if not have and os.path.isdir(DRIVE_SYNTH):
    print(f'Syncing synthetic data from {DRIVE_SYNTH} ...')
    !mkdir -p {LOCAL_SYNTH}
    !rsync -a {DRIVE_SYNTH}/ {LOCAL_SYNTH}/
    have = all(os.path.exists(os.path.join(LOCAL_SYNTH, f)) for f in need)

if not have:
    raise FileNotFoundError(
        f'Missing synthetic CSVs in {LOCAL_SYNTH} (need {need}). '
        f'Include data/synthetic/ in AnyMatch.zip or upload to {DRIVE_SYNTH}.')

import pandas as pd
for f in need:
    lab = pd.read_csv(os.path.join(LOCAL_SYNTH, f), usecols=['label'])['label'].value_counts().to_dict()
    print(f'  {f:28s} label dist {lab}')

  finetune_train_v1.csv        label dist {0: 17267, 1: 13586}
  finetune_test_v1.csv         label dist {0: 6729, 1: 2409}
  realistic_eval_v1.csv        label dist {0: 9000, 1: 1000}


## 4. Stage the base checkpoint to resume from

The fine-tune resumes from the all9 mode4 checkpoint. Pull it from Drive (where `anymatch_training.ipynb` backed it up) into the local `saved_models/` dir.

In [4]:
BASE_CKPT  = 'saved_models/anymatch_all9_gpt2_mode4'
DRIVE_BASE = f'{DRIVE_CKPT_DIR}/anymatch_all9_gpt2_mode4'

if not os.path.exists(os.path.join(BASE_CKPT, 'config.json')):
    assert os.path.isdir(DRIVE_BASE), (
        f'Base checkpoint not found locally or at {DRIVE_BASE}. '
        'Run anymatch_training.ipynb first (it backs the checkpoint up to Drive).')
    print(f'Staging base checkpoint from {DRIVE_BASE} ...')
    !mkdir -p {BASE_CKPT}
    !rsync -a {DRIVE_BASE}/ {BASE_CKPT}/

print('Base checkpoint contents:')
!ls -la {BASE_CKPT}

Base checkpoint contents:
total 487564
drwxr-xr-x 2 root root      4096 May 26 16:49 .
drwxr-xr-x 4 root root      4096 May 26 21:48 ..
-rw-rw-r-- 1 root root       962 May 26 14:26 config.json
-rw-rw-r-- 1 root root    456318 May 26 14:26 merges.txt
-rw-rw-r-- 1 root root 497780432 May 26 14:26 model.safetensors
-rw-rw-r-- 1 root root       438 May 26 14:26 special_tokens_map.json
-rw-rw-r-- 1 root root       545 May 26 14:26 tokenizer_config.json
-rw-rw-r-- 1 root root    999186 May 26 14:26 vocab.json


## 5. Run the fine-tune

`finetune_alliance.py` resumes from `BASE_CKPT`, serializes the synthetic CSVs via the shared `alliance_schema` (mode4, three name fields), trains gently, and early-stops on `finetune_test`.

**Gentle defaults** (recommended): `--lr 1e-5`, `--epochs 10`, `--patience 3`. These minimise catastrophic forgetting of the general EM ability baked into the base checkpoint. On an A100, ~30k pairs × ≤10 epochs is roughly 10–25 min.

The `--eval_csv` flag also prints a final realistic-eval (1:9) F1 — that's the number that reflects the real operating distribution.

In [5]:
FT_CKPT = 'saved_models/anymatch_alliance_ft_v1'

!python finetune_alliance.py \
    --base_checkpoint {BASE_CKPT} \
    --train_csv data/synthetic/finetune_train_v1.csv \
    --valid_csv data/synthetic/finetune_test_v1.csv \
    --eval_csv  data/synthetic/realistic_eval_v1.csv \
    --save_model_path {FT_CKPT} \
    --base_model gpt2 \
    --serialization_mode mode4 \
    --lr 1e-5 \
    --epochs 10 \
    --patience 3 \
    --train_batch_size 64 \
    --seed 42

Resuming from checkpoint: saved_models/anymatch_all9_gpt2_mode4
Loading weights: 100% 149/149 [00:00<00:00, 5545.56it/s]
Serializing train / valid ...
0 out of 30853 data samples are filtered.
0 out of 9138 data samples are filtered.
Train pairs: 30853 | Valid pairs: 9138
Config — lr: 1e-05 | epochs: 10 | patience: 3 (start 0) | batch: 64 | mode: mode4 | max_len: 350
Epoch: 1 | Train Loss: 0.0221 | Valid Loss: 0.0097 | Train acc: 99.28 | Valid acc: 99.75 | Train f1: 99.19 | Valid f1: 99.53 | Train Time: 315.3228325843811 secs
The best model is updated at epoch: 1 with f1 score 0.9953488372093023
Epoch: 2 | Train Loss: 0.0022 | Valid Loss: 0.0060 | Train acc: 99.94 | Valid acc: 99.90 | Train f1: 99.93 | Valid f1: 99.82 | Train Time: 313.1540756225586 secs
The best model is updated at epoch: 2 with f1 score 0.9981684981684982
Epoch: 3 | Train Loss: 0.0025 | Valid Loss: 0.0043 | Train acc: 99.94 | Valid acc: 99.95 | Train f1: 99.94 | Valid f1: 99.91 | Train Time: 312.964462518692 secs
The

## 6. Back up the fine-tuned checkpoint to Drive

So it survives runtime disconnect and can be downloaded for local inference.

In [6]:
!rsync -a {FT_CKPT}/ {DRIVE_CKPT_DIR}/anymatch_alliance_ft_v1/
!ls {DRIVE_CKPT_DIR}/anymatch_alliance_ft_v1/

config.json		 model.safetensors	tokenizer.json
finetune_train_stat.csv  tokenizer_config.json


## 7. Regression check — did general EM ability survive?

Sequential fine-tuning risks catastrophic forgetting. Score the fine-tuned model on a public test split (e.g. `abt_buy`) and compare against the base checkpoint\'s F1 from training. A large drop means the LR/epochs were too aggressive — lower `--lr` or `--patience` and re-run.

In [7]:
import pandas as pd
from transformers import GPT2Tokenizer, GPT2ForSequenceClassification
from data import GPTDataset
from utils.data_utils import read_single_row_data
from utils.train_eval import inference

def score_public(ckpt, dataset='abt'):
    tok = GPT2Tokenizer.from_pretrained(ckpt)
    m = GPT2ForSequenceClassification.from_pretrained(ckpt)
    m.config.pad_token_id = m.config.eos_token_id
    _, _, test_df = read_single_row_data(f'data/prepared/{dataset}', mode='mode4', print_info=False)
    test_d = GPTDataset(tok, test_df, max_len=10000)
    f1, acc = inference(tok, m, test_d, batch_size=128, base_model='gpt2')
    return f1, acc

for name, ckpt in [('base (all9)', BASE_CKPT), ('fine-tuned', FT_CKPT)]:
    f1, acc = score_public(ckpt, 'abt')
    print(f'{name:14s} abt_buy  F1={f1*100:.2f}  acc={acc*100:.2f}')

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

0 out of 1916 data samples are filtered.
The predictions and ground truth are:
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

0 out of 1916 data samples are filtered.
The predictions and ground truth are:
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

## 8. Next steps

- **Inference**: download `anymatch_alliance_ft_v1/` to `AnyMatch/saved_models/` locally, then point `CKPT_DIR` in `anymatch_synthetic_inference.ipynb` / `anymatch_alliance_inference.ipynb` at it. Both already serialize via `alliance_schema`, so they match this checkpoint.
- **Iterate**: read the per-`case_type` breakdown on `realistic_eval` (group the predictions by the `case_type` column). Scale up only the under-performing scenario buckets in `generate_synthetic.py`, regenerate, and re-fine-tune — cheap because generation is ~4 s.
- **Forgetting too high?** Lower `--lr` (e.g. 5e-6) or `--patience 2`; consider freezing lower GPT-2 layers as a v2 knob.